# 02 — Qwen3-8B component ablations

Run the **preregistered single-mechanism ablations** (RQ3, exploratory): one
mechanism disabled per row, each through the same campaign CLI with
`--condition ablation:<name>`, plus the allow-all control and the full-defence
rows.

The exact switch values the service accepts are the four members of
`backend/aegisgraph/engine.py::Ablation`:

| Row | `--condition` | Disables |
| --- | --- | --- |
| control | `control` | (allow-all baseline) |
| full defence | `defence` | nothing |
| ABL-1 | `ablation:provenance_enforcement` | the provenance labels for the evaluation |
| ABL-2 | `ablation:trust_ceiling` | the caller's trust ceiling |
| ABL-3 | `ablation:strict_confirmation` | the strict confirmation channel |
| ABL-4 | `ablation:rewrite_revalidation` | the rewrite re-evaluation |

The ablation is a **process-level** environment variable (`AEGISGRAPH_ABLATION`),
so this notebook starts one ablated gateway per row on loopback; the campaign CLI
never manages processes. An ablated run is never reported as the defence.

**No real-model result exists in this repository.** This notebook writes no output
of its own.

Run this notebook from a checkout of the repository (see `notebooks/README.md`); it locates the repository root automatically and `chdir`s into it. Nothing is
committed: the run directories and bundles stay in the cloud.


In [ ]:
import hashlib, json, os, re, subprocess, sys, tempfile, time, urllib.request, zipfile
from pathlib import Path


def _repo_root(start):
    for candidate in (start, *start.parents):
        if (candidate / "scripts" / "bench_campaign.py").is_file() or (
            candidate / "scripts" / "bench_validate.py"
        ).is_file():
            return candidate
    return start


configured_repo = os.environ.get("AEGISGRAPH_REPO")
REPO = (Path(configured_repo).expanduser() if configured_repo else _repo_root(Path.cwd())).resolve()
if not (REPO / "scripts" / "bench_campaign.py").is_file():
    cloud_root = Path("/content") if Path("/content").is_dir() else Path("/kaggle/working")
    if not cloud_root.is_dir():
        cloud_root = Path.cwd()
    REPO = cloud_root / "IndabaX26"
    if REPO.exists():
        raise RuntimeError("cloud checkout exists but is not AegisGraph: %s" % REPO)
    repo_ref = os.environ.get("AEGISGRAPH_REF", "main")
    subprocess.run(["git", "clone", "--branch", repo_ref, "--depth", "1",
                    "https://github.com/OussemaHarrabi/IndabaX26.git", str(REPO)], check=True)
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

SCRATCH = Path(os.environ.get("AEGISGRAPH_SCRATCH") or tempfile.mkdtemp(prefix="aegisgraph-notebook-"))
SCRATCH.mkdir(parents=True, exist_ok=True)
ISSUER_DIR = SCRATCH / "dev-issuer"
POLICY_ADMIN_TOKEN = SCRATCH / "policy-admin.jwt"
DECISION_TOKEN = SCRATCH / "decision.jwt"
GATEWAY = {"proc": None, "url": None, "ablation": None}


def sh(args, *, check=True, capture=False, env=None):
    """Run a command from the repository root and echo it.

    ``capture=True`` returns the completed process with its output printed, so a
    notebook cell never silently swallows a CLI's diagnostics.
    """
    printable = [str(item) for item in args]
    print("$", " ".join(printable))
    proc = subprocess.run(printable, cwd=str(REPO), text=True, capture_output=capture, env=env)
    if capture:
        if proc.stdout:
            print(proc.stdout)
        if proc.stderr:
            print(proc.stderr, file=sys.stderr)
    if check and proc.returncode != 0:
        raise RuntimeError("command failed (exit %d): %s" % (proc.returncode, " ".join(printable)))
    return proc


def run_dirs_from(output):
    """The run directories a campaign invocation printed."""
    return [Path(match) for match in re.findall(r"run directory:\s*(\S+)", output)]


def init_issuer():
    if not (ISSUER_DIR / "jwks.json").is_file():
        sh([sys.executable, "scripts/dev_issuer.py", "init", "--directory", ISSUER_DIR], capture=True)
    for subject, role, path in (
        ("bench-policy-admin", "policy_admin", POLICY_ADMIN_TOKEN),
        ("bench-decision-client", "decision_client", DECISION_TOKEN),
    ):
        minted = subprocess.run(
            [sys.executable, "scripts/dev_issuer.py", "mint", "--directory", ISSUER_DIR,
             "--tenant", "bench", "--subject", subject, "--role", role, "--ttl", "43200"],
            cwd=str(REPO), text=True, capture_output=True, check=True,
        )
        path.write_text(json.loads(minted.stdout)["token"], encoding="utf-8")
    print("issuer ready:", ISSUER_DIR)


def stop_gateway():
    proc = GATEWAY.get("proc")
    if proc is not None and proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=15)
        except Exception:
            proc.kill()
    GATEWAY.update(proc=None, url=None, ablation=None)


def start_gateway(port=8091, ablation=None):
    """Start the frozen gateway on loopback; optionally as an ablated build.

    The campaign CLI never manages processes and never sets AEGISGRAPH_*, so the
    notebook owns the gateway lifecycle. An ablation is a process-level switch.
    """
    stop_gateway()
    env = dict(os.environ)
    env.update({
        "PYTHONPATH": "backend",
        "AEGISGRAPH_AUTH_MODE": "required",
        "AEGISGRAPH_JWT_ISSUER": "https://dev-issuer.aegisgraph.local",
        "AEGISGRAPH_JWT_AUDIENCE": "aegisgraph",
        "AEGISGRAPH_JWKS": str(ISSUER_DIR / "jwks.json"),
    })
    if ablation:
        env["AEGISGRAPH_ABLATION"] = ablation
    log_path = SCRATCH / ("gateway-%s.log" % (ablation or "defence"))
    log = log_path.open("w", encoding="utf-8")
    proc = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "aegisgraph.app:app",
         "--host", "127.0.0.1", "--port", str(port)],
        cwd=str(REPO), env=env, stdout=log, stderr=subprocess.STDOUT,
    )
    url = "http://127.0.0.1:%d" % port
    GATEWAY.update(proc=proc, url=url, ablation=ablation)
    for _ in range(120):
        if proc.poll() is not None:
            raise RuntimeError("gateway exited early; see %s" % log_path)
        try:
            with urllib.request.urlopen(url + "/healthz", timeout=2) as response:
                if json.load(response).get("status") == "ok":
                    print("gateway ready at", url, "(ablation: %s)" % (ablation or "none"))
                    return url
        except Exception:
            time.sleep(0.5)
    raise TimeoutError("gateway did not become healthy; see %s" % log_path)


def publish_policies(url):
    sh([sys.executable, "scripts/bench_policies.py", "publish",
        "--defense-url", url, "--token-file", POLICY_ADMIN_TOKEN], capture=True)


def adapter_json(**overrides):
    """The frozen QwenConfig as the driver's inline --adapter-json value.

    The CLI takes the JSON itself (``--adapter-json '{"revision": ...}'``), not a
    path; a copy is also written to the scratch directory for the record.
    """
    from benchmark.qwen import QwenConfig
    payload = json.dumps(QwenConfig(**overrides).to_json(), sort_keys=True)
    (SCRATCH / "adapter.json").write_text(payload + "\n", encoding="utf-8")
    print(payload)
    return payload


def campaign(stage, condition, seeds, *, defense_url=None, adapter_json=None,
             resume=False, bundle=False, max_scenarios=None, dry_run=False):
    """Invoke the campaign CLI. The CLI owns all evaluation logic."""
    args = [sys.executable, "scripts/bench_campaign.py", "--stage", stage,
            "--condition", condition, "--model", MODEL, "--runs-dir", str(RUNS_DIR)]
    for seed in seeds:
        args += ["--seed", str(seed)]
    if defense_url:
        args += ["--defense-url", defense_url]
    if dry_run:
        args += ["--dry-run"]
    else:
        if MODEL == "qwen" and adapter_json is not None:
            args += ["--adapter-json", str(adapter_json)]
        if DECISION_TOKEN.is_file():
            args += ["--auth-token-file", str(DECISION_TOKEN)]
        if resume:
            args += ["--resume"]
        if max_scenarios is not None:
            args += ["--max-scenarios", str(max_scenarios)]
        if bundle:
            args += ["--bundle"]
    return sh(args, capture=True)


def score_run(run_dir):
    """Score a run directory into score.json/score.txt (protocol section 8)."""
    run_dir = Path(run_dir)
    table = sh([sys.executable, "scripts/bench_score.py", "--run", run_dir], capture=True).stdout
    (run_dir / "score.txt").write_text(table, encoding="utf-8")
    sh([sys.executable, "scripts/bench_score.py", "--run", run_dir, "--json",
        "--out", str(run_dir / "score.json")], capture=True)
    print("scored:", run_dir.name)


def verify_zip(zip_path):
    zip_path = Path(zip_path)
    digest = hashlib.sha256(zip_path.read_bytes()).hexdigest()
    with zipfile.ZipFile(zip_path) as archive:
        corrupt = archive.testzip()
    print("bundle:", zip_path.name, "sha256:", digest, "integrity:", "ok" if corrupt is None else corrupt)
    return digest


In [ ]:
# @title Parameters
ABLATIONS = ["provenance_enforcement", "trust_ceiling", "strict_confirmation", "rewrite_revalidation"]

# The ablation matrix is exploratory; the stage and seed list are fixed by the
# freeze record before the row runs. Stage B/C require >= 3 preregistered seeds,
# so DEFAULT_SEEDS = [1729] (the anchor) is deliberately below the floor: the CLI's
# dry run refuses it until the frozen list is entered here. That refusal is the
# preregistration guard, not a bug.
STAGE = "B"
SEEDS = [1729, 2741, 3253, 4253, 5527]
MODEL = "qwen"
RUNS_DIR = Path(os.environ.get("AEGISGRAPH_RUNS_DIR", str(SCRATCH / "runs")))
GATEWAY_PORT = 8091
DEVICE = "cuda"

MODEL_REVISION = os.environ.get("AEGISGRAPH_MODEL_REVISION", "unpinned")
QUANTIZATION = "4bit"
DTYPE = "bfloat16"
TEMPERATURE = 0.0
TOP_P = 1.0
MAX_NEW_TOKENS = 768
THINKING = False
FORMAT_RETRIES = 1
print("stage:", STAGE, "seeds:", SEEDS, "ablations:", ABLATIONS)


## 1. Runtime, toolchain and model

The same checks as notebook 00, condensed.


In [ ]:
GPU_NAME, VRAM_GB = None, None
try:
    import torch
    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        GPU_NAME, VRAM_GB = props.name, props.total_memory / 1024 ** 3
        print("GPU:", GPU_NAME, "| VRAM (GiB):", round(VRAM_GB, 1))
except Exception as error:
    print("torch is not importable here:", error)

ENV_OK = GPU_NAME is not None and VRAM_GB is not None and VRAM_GB >= 14.0
print("environment suitable:", ENV_OK)

if ENV_OK:
    import shlex
    from benchmark.qwen import INSTALL_COMMAND, QwenConfig, QwenModelAdapter
    command = shlex.split(INSTALL_COMMAND)
    command[0] = sys.executable
    sh(command, check=False)
    adapter = QwenModelAdapter(QwenConfig(
        revision=MODEL_REVISION, quantization=QUANTIZATION, dtype=DTYPE,
        temperature=TEMPERATURE, top_p=TOP_P, max_new_tokens=MAX_NEW_TOKENS,
        thinking=THINKING, device=DEVICE, format_retries=FORMAT_RETRIES,
    ))
    adapter.load()
    resolved_identity = adapter.identity().to_json()["parameters"]
    MODEL_REVISION = resolved_identity.get("resolved_revision") or MODEL_REVISION
    if MODEL_REVISION == "unpinned":
        raise RuntimeError("the exact Hugging Face model revision could not be resolved")
    if not resolved_identity.get("chat_template_sha256"):
        raise RuntimeError("the tokenizer chat-template digest could not be resolved")
    if not resolved_identity.get("max_context_tokens"):
        raise RuntimeError("the model context limit could not be resolved")
    print(json.dumps(resolved_identity, indent=2, sort_keys=True))
    import gc
    del adapter
    gc.collect()
    torch.cuda.empty_cache()
ADAPTER_OK = ENV_OK


In [ ]:
validated = sh([sys.executable, "scripts/bench_validate.py"], capture=True).stdout
if "RESULT: PASS" not in validated:
    raise RuntimeError("the dataset did not validate; no row may run")
seal = sh([sys.executable, "scripts/bench_seal.py", "verify"], capture=True).stdout
if '"ciphertext_matches": true' not in seal:
    raise RuntimeError("the seal did not verify; no row may run")


## 2. Dry-run every row's plan first

A dry run opens no socket and loads no model; it validates the plan (including the
seed floor). If any row is refused, stop and close the configuration first.


In [ ]:
for condition in ["control", "defence"] + ["ablation:" + name for name in ABLATIONS]:
    check = campaign(STAGE, condition, SEEDS, defense_url="http://127.0.0.1:%d" % GATEWAY_PORT, dry_run=True)
    if "RESULT: DRY-RUN" not in check.stdout:
        raise RuntimeError("the plan for %s was refused; close the configuration first" % condition)


## 3. Control and full-defence rows

A plain gateway (no ablation) serves both rows.


In [ ]:
init_issuer()
GATEWAY_URL = start_gateway(GATEWAY_PORT)
publish_policies(GATEWAY_URL)
ADAPTER_JSON = adapter_json(
    revision=MODEL_REVISION, quantization=QUANTIZATION, dtype=DTYPE,
    temperature=TEMPERATURE, top_p=TOP_P, max_new_tokens=MAX_NEW_TOKENS,
    thinking=THINKING, device=DEVICE, format_retries=FORMAT_RETRIES,
)


In [ ]:
ALL_RUNS = []
for condition in ["control", "defence"]:
    completed = campaign(STAGE, condition, SEEDS, defense_url=GATEWAY_URL, adapter_json=ADAPTER_JSON)
    ALL_RUNS += run_dirs_from(completed.stdout)
print("control + defence run directories:", [p.name for p in ALL_RUNS])


## 4. Ablation rows

Each row restarts the gateway as that ablated build (its `GET /api/v1/version`
reports the active ablation before any request), publishes the policy sets and
runs the same CLI condition.


In [ ]:
for name in ABLATIONS:
    ablated_url = start_gateway(GATEWAY_PORT, ablation=name)
    with urllib.request.urlopen(ablated_url + "/api/v1/version", timeout=5) as response:
        version = json.load(response)
    if version.get("ablation") != name:
        raise RuntimeError("the gateway did not report ablation %s" % name)
    publish_policies(ablated_url)
    completed = campaign(STAGE, "ablation:" + name, SEEDS,
                         defense_url=ablated_url, adapter_json=ADAPTER_JSON)
    ALL_RUNS += run_dirs_from(completed.stdout)
print("all run directories:", [p.name for p in ALL_RUNS])


## 5. Score the rows

Scoring is `scripts/bench_score.py`; the driver never scores. `score.json` is
required before analysis, which reproduces it.


In [ ]:
for run_dir in ALL_RUNS:
    score_run(run_dir)


## 6. Analysis handoff

The ablation family is exploratory: it is analysed by the paired machinery with
Benjamini–Hochberg at q = 0.05, labelled exploratory. Use **notebook 03** with
`EXPLORATORY = True`, or the CLI directly:

```bash
python scripts/bench_analyze.py --run <ablation-run> --control <full-defence-run> \
  --out analysis-out --exploratory
```

An ablated run is never reported as the defence, and any change is compared with
the declared direction; a zero change is a null effect, not a retry.


In [ ]:
stop_gateway()
print("ablated gateways stopped; run directories are under:", RUNS_DIR)
